# Mini ML Use Case: End-to-End Business Intelligence Pipeline 📊
### *Predicting Customer Churn for Telecom Operations*

## 1. Business Problem Understanding

### Problem Statement
Customer attrition (Churn) is one of the biggest challenges in the telecom industry. The cost of acquiring a new customer is significantly higher than retaining an existing one. Identifying customers at risk of churning allows the business to intervene proactively.

### Business Objectives
- **Predictive Analytics**: Develop a machine learning classification model to identify customers likely to churn.
- **Root Cause Analysis**: Use Explainable AI (XAI) to uncover the key drivers of churn.
- **Actionable Insights**: Provide data-driven strategic recommendations to reduce churn rates.

### Expected Business Value & KPIs
- **Increase Retention**: Targeting high-risk customers with customized offers.
- **Reduce CAC (Customer Acquisition Cost)**: By preserving the existing customer base, overall operational costs decrease.
- **KPIs**: Model ROC-AUC, Churn Rate Reduction %, Campaign Conversion Rate.


## 2. Data Collection & Understanding
We load our customer demographic and service data, analyzing its structure and identifying missing information.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('../data/raw/customer_churn.csv')
display(df.head())


In [ ]:
print(f"Dataset Shape: {df.shape}")
print("\nMissing Values:")
print(df.isnull().sum())
print("\nSummary Statistics:")
display(df.describe())


## 3. Exploratory Data Analysis (EDA)
Visual storytelling to uncover behavioral trends and operational insights.


In [ ]:
# 1. Target Variable Analysis
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Customer Churn Distribution')
plt.show()

# 2. Tenure Trends (When do people churn?)
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='Tenure', hue='Churn', fill=True, common_norm=False, palette='Set1')
plt.title('Tenure Distribution by Churn Status (New customers churn faster)')
plt.show()

# 3. Contract Type Influence
plt.figure(figsize=(8, 5))
sns.countplot(x='Contract', hue='Churn', data=df, palette='viridis')
plt.title('Churn by Contract Type (Month-to-month is high risk)')
plt.show()


## 4. Feature Engineering & Preprocessing
Creating meaningful business features and deploying robust Scikit-Learn Pipelines.

**Engineered Features:**
- **TenureGroup**: Segmentation based on customer lifespan.
- **SpendingCategory**: Quantile-based spending buckets.
- **IsHighRisk**: A custom boolean flag for (Month-to-Month + Fiber Optic + No Tech Support).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df_clean = df.copy()

# Feature Engineering
bins = [0, 12, 24, 48, 60, 100]
labels = ['0-1 Year', '1-2 Years', '2-4 Years', '4-5 Years', '5+ Years']
df_clean['TenureGroup'] = pd.cut(df_clean['Tenure'], bins=bins, labels=labels, right=False).astype(str)
df_clean['SpendingCategory'] = pd.qcut(df_clean['MonthlyCharges'], 4, labels=['Low', 'Medium', 'High', 'Premium']).astype(str)
is_high_risk = (df_clean['Contract'] == 'Month-to-month') & (df_clean['InternetService'] == 'Fiber optic') & (df_clean['TechSupport'] == 'No')
df_clean['IsHighRisk'] = is_high_risk.astype(int)

# Split data
X = df_clean.drop(columns=['CustomerID', 'Churn'])
y = df_clean['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Pipeline
num_feats = ['Tenure', 'MonthlyCharges', 'TotalCharges']
cat_feats = ['Gender', 'Partner', 'InternetService', 'TechSupport', 'Contract', 'TenureGroup', 'SpendingCategory']
pass_feats = ['SeniorCitizen', 'IsHighRisk']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_feats),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_feats),
        ('pass', 'passthrough', pass_feats)
    ])


## 5. Model Training & Hyperparameter Tuning
We train Logistic Regression and XGBoost, tuning XGBoost to maximize performance.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Convert Target for XGBoost (0/1)
y_train_num = y_train.map({'No': 0, 'Yes': 1})
y_test_num = y_test.map({'No': 0, 'Yes': 1})

# Logistic Regression
lr_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', LogisticRegression(random_state=42))])
lr_pipe.fit(X_train, y_train_num)

# XGBoost Tuning
xgb_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', xgb.XGBClassifier(random_state=42, eval_metric='logloss'))])
xgb_grid = GridSearchCV(xgb_pipe, {'model__n_estimators': [100, 150], 'model__learning_rate': [0.05, 0.1]}, cv=3, scoring='roc_auc', n_jobs=-1)
xgb_grid.fit(X_train, y_train_num)
best_xgb = xgb_grid.best_estimator_

models = {'Logistic Regression': lr_pipe, 'XGBoost (Tuned)': best_xgb}
print(f"XGBoost Tuned Best Params: {xgb_grid.best_params_}")


## 6. Evaluation Metrics
We compare models using ROC-AUC, Accuracy, Precision, and Recall.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_num, y_pred),
        'ROC_AUC': roc_auc_score(y_test_num, y_proba)
    })
    
    cm = confusion_matrix(y_test_num, y_pred)
    plt.figure(figsize=(5, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Stayed', 'Churned'], yticklabels=['Stayed', 'Churned'])
    plt.title(f'Confusion Matrix: {name}')
    plt.show()

display(pd.DataFrame(results))


## 7. Explainable AI (SHAP) & Business Insights
We deploy SHAP to decode the blackbox model.

### Key Business Drivers:
- **Contract Type**: Month-to-month contracts strongly drive churn.
- **Tenure**: Low tenure strongly drives churn.
- **Risk Flag**: Our engineered `IsHighRisk` feature is highly predictive.


In [ ]:
import shap

# Transform data for SHAP
X_sample = X_train.sample(500, random_state=42)
X_trans = best_xgb.named_steps['preprocessor'].transform(X_sample)

# Feature Names
cat_names = best_xgb.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_feats)
all_feats = num_feats + list(cat_names) + pass_feats

explainer = shap.TreeExplainer(best_xgb.named_steps['model'])
shap_values = explainer.shap_values(X_trans)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_trans, feature_names=all_feats, show=False)
plt.title('SHAP Feature Importance Summary')
plt.show()


## 8. Strategic Business Recommendations
Based on the data and model interpretation, we recommend the following strategic actions:

1. **Incentivize Long-Term Contracts**: Since Month-to-Month contracts are the primary driver of churn, offer a 10-15% discount for customers who upgrade to 1-Year or 2-Year plans.
2. **First 90-Day Retention Focus**: The KDE distribution showed massive churn in the first few months. Implement a dedicated onboarding and check-in sequence during the first 90 days.
3. **Tech Support Bundling**: Fiber optic customers without tech support have disproportionate churn rates (likely due to frustration). Bundle Tech Support automatically at a reduced rate for high-tier internet users.
4. **Deploy Prediction Pipeline**: Use the `predict.py` script in a weekly batch job. Any customer flagged with > 65% churn probability should automatically be routed to the retention team for proactive outreach.
